In [35]:
import numpy as np
import pandas as pd

In [36]:
# new optimization procedure (2023.04.18)
def monotonization(input):

  import numpy as np
  n = len(input)
  c = np.nanmax(input)
  # k = n - np.isnan(input).sum()
  input_obs = input[~np.isnan(input)]
  obs = len(input_obs)
  
  ind = np.where(1-np.isnan(input))
  K = ind[0]

  import gurobipy as gp
  m = gp.Env(empty=True)
  m.setParam('WLSACCESSID', '54ea4edc-45f3-4ac2-b06f-cc4809a3c06c')
  m.setParam('WLSSECRET', '2802dfc5-13c3-4ceb-b6c2-b217b0d5994a')
  m.setParam('LICENSEID', 947226)
  m.setParam("OutputFlag", 0) #add by CHP
  m.start()

# Create the model within the Gurobi environment
  m = gp.Model(env=m)

  from gurobipy import GRB

  y = m.addVars(obs, lb=1, ub=c, name="y")
  u = m.addVars(obs, lb=0, name="u")
  w = m.addVars(obs, lb=0, name="w")

  m.setObjective(gp.quicksum(u[i]+w[i] for i in range(obs)), GRB.MINIMIZE)

  m.addConstr(1 <= y[0])
  m.addConstr(y[obs-1] <= c)
  m.addConstrs(y[i] <= y[i+1] for i in range(obs-1))
  m.addConstrs(y[j]-input_obs[j] == u[j]-w[j] for j in range(obs))

  m.optimize()

  y_out = m.getAttr('X', y)

  x_out = input

  # print(x_out)
  # print(y_out)
  # print(K)

  # for j in range(n):
  #   if j <= (K[0]+K[1])/2:
  #     x_out[j] = y_out[0]
  #   elif j > (K[obs-2]+K[obs-1])/2:
  #     x_out[j] = y_out[obs-1]
  #   else:
  #     for i in range(1,obs-1):
  #       if (K[i-1]+K[i])/2 < j <= (K[i]+K[i+1])/2:
  #         x_out[j] = y_out[i]
  
  for j in range(n):
    if j <= K[0]:
      x_out[j] = y_out[0]
    for k in range(obs-1):
      if j >= K[k] and j < K[k+1]:
        x_out[j] = y_out[k+1]
      if j >= K[k+1]:
        x_out[j] = y_out[obs-1]
      
  for i in range(obs):
    x_out[K[i]] = input_obs[i]

  return x_out


In [103]:
def rmse_cal2(df, power):
  avail_ind = np.where(1-df.isna())
  avail_r_ind = avail_ind[0]
  avail_c_ind = avail_ind[1]
  l = len(avail_r_ind)

  masked_df = np.ma.masked_array(df, np.isnan(df))
  df_out = df.copy()

  w = np.array(df.corr())
  w[w<0]=0
  for i in range(len(w)):
    w[i,i]=0

  corr_mat = w

  for k in range(l):
    i = avail_r_ind[k]
    j = avail_c_ind[k]
    df_out.iloc[i,j] = np.ma.average(masked_df[i,:], weights=np.power(corr_mat[:,j]+1e-10,power))
  diff = df - df_out
  rmse = ((diff ** 2).sum().sum()/l) ** 0.5
  return rmse

In [38]:
def nested_quantile(df, a_min, a_max, n_level, RMSE_max):
  import numpy as np
  Q = np.zeros(5)
  Q[0] = a_min
  Q[4] = a_max
  beta = abs(Q[0]-Q[4])/4
  for j in range(1,4):
    Q[j] = Q[0] + j*beta

  # RMSE_L = rmse_calculator(df,Q[0])
  # RMSE_R = rmse_calculator(df,Q[4])
  RMSE_L = rmse_cal2(df,Q[0])
  RMSE_R = rmse_cal2(df,Q[4])

  # When RMSE_M=1, it is temporary
  RMSE_M = 1

  for i in range(n_level):

    RMSE = pd.DataFrame([[Q[0],RMSE_L],[Q[1],None],[Q[2],RMSE_M],[Q[3],None],[Q[4],RMSE_R]])

    for k in range(1,4):
      if k != 2 or RMSE_M == 1:
        # RMSE.iloc[k,1] =  rmse_calculator(df,Q[k])
        RMSE.iloc[k,1] =  rmse_cal2(df,Q[k])
      else:
        RMSE.iloc[k,1] = RMSE_M
    
    RMSE_sorted = RMSE.sort_values(by=[1])
        
    #print('RMSE Sorted:')
    #display(RMSE_sorted)
    
    alpha = RMSE_sorted.iloc[0,0]

    if RMSE_sorted.iloc[0,0] == Q[0]:
      Q[4] = Q[1]
      RMSE_R = RMSE.iloc[1,1] 
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[4]:
      Q[0] = Q[3] 
      RMSE_L = RMSE.iloc[3,1]
      RMSE_M = 1
    elif RMSE_sorted.iloc[0,0] == Q[1]:
      Q[4] = Q[2]
      RMSE_M = RMSE.iloc[1,1]
      RMSE_R = RMSE.iloc[2,1]
    elif RMSE_sorted.iloc[0,0] == Q[2]:
      Q[0] = Q[1]
      Q[4] = Q[3]
      RMSE_L = RMSE.iloc[1,1]
      RMSE_M = RMSE.iloc[2,1]
      RMSE_R = RMSE.iloc[3,1]
    elif RMSE_sorted.iloc[0,0] == Q[3]:
      Q[0] = Q[2]
      RMSE_L = RMSE.iloc[2,1]
      RMSE_M = RMSE.iloc[3,1]

    beta = abs(Q[0]-Q[4])/4
    for j in range(1,4):
      Q[j] = Q[0] + j*beta

  return alpha

In [74]:
item_ids = [0,1,2,3,4]  # 가정한 item_id 리스트
user_ids = [0,1,2,3]  # 가정한 user_id 리스트

# CSV 파일을 읽어들입니다. 첫 행과 첫 열에 헤더가 없음을 명시합니다.
# names 매개변수로 열 이름을 지정합니다.
df = pd.read_csv("toy_example.csv", header=None, index_col=None, names=user_ids)

# 행 인덱스(item_id)를 수동으로 설정합니다.
df.index = item_ids

print(df)

     0    1    2    3
0  1.0  NaN  1.0  NaN
1  2.0  3.0  1.0  3.0
2  NaN  2.0  NaN  2.0
3  4.0  NaN  3.0  NaN
4  5.0  1.0  5.0  5.0


In [75]:
def normal1(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = (df[i]-1)/(maxs[i]-1)
  return df_out#, maxs

In [76]:
def normal2(df):
  n = df.shape[1]
  maxs = [df[i].max() for i in range(n)]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]/maxs[i]
  return df_out#, maxs

In [77]:
def normal3(df):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  maxs = [int(df[i].max()) for i in range(n)]
  probs = {}
  for j in range(n):
    mm = m - df.isna().sum(axis=0)[j]
    probs[j] = [np.count_nonzero(df[j] == k)/mm for k in range(1,maxs[j]+1)]
  df_out = df.copy()
  for j in range(n):
    if maxs[j] == 2:
      df_out[j] = df[j]-1
    else:
      c = maxs[j]
      for i in range(m):
        r = df.iloc[i,j]
        if np.isnan(r):
          df_out.iloc[i,j] = np.nan
        elif r == 1:
          df_out.iloc[i,j] = 0
        elif r == 2:
          df_out.iloc[i,j] = (probs[j][0]+probs[j][1])/(sum(probs[j][:c-1])+sum(probs[j][1:c]))
        else:
          df_out.iloc[i,j] = (sum(probs[j][:int(r)-1])+sum(probs[j][1:int(r)]))/(sum(probs[j][:int(c)-1])+sum(probs[j][1:int(c)]))
  return df_out, maxs, probs

In [78]:
normal1(df)

,0,1,2,3
0,0.00,NaN,0.0,NaN
1,0.25,1.0,0.0,0.50
2,NaN,0.5,NaN,0.25
3,0.75,NaN,0.5,NaN
4,1.00,0.0,1.0,1.00


In [79]:
normal2(df)

,0,1,2,3
0,0.2,NaN,0.2,NaN
1,0.4,1.000000,0.2,0.6
2,NaN,0.666667,NaN,0.4
3,0.8,NaN,0.6,NaN
4,1.0,0.333333,1.0,1.0


In [80]:
df_out, maxs, probs = normal3(df)
display (df_out)
display (maxs)
display (probs)

,0,1,2,3
0,0.000000,NaN,0.0,NaN
1,0.333333,1.0,0.0,0.6
2,NaN,0.5,NaN,0.2
3,0.666667,NaN,0.6,NaN
4,1.000000,0.0,1.0,1.0


[5, 3, 5, 5]

{0: [0.25, 0.25, 0.0, 0.25, 0.25],
 1: [0.3333333333333333, 0.3333333333333333, 0.3333333333333333],
 2: [0.5, 0.0, 0.25, 0.0, 0.25],
 3: [0.0, 0.3333333333333333, 0.3333333333333333, 0.0, 0.3333333333333333]}

In [81]:
def unnormal3(df,maxs,probs):
  def T2(x, prob, max):
    y = (2*x-prob[0])/(sum(prob[:max-1])+sum(prob[1:]))
    return y
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for j in range(n):
    prob = probs[j]
    max = maxs[j]
    if maxs[j] == 2:
      for i in range(m):
        if df.iloc[i,j] < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        else:
          df_out.iloc[i,j] = 2
    else:
      for i in range(m):
        r = df.iloc[i,j]
        if r < T2(prob[0],prob,max):
          df_out.iloc[i,j] = 1
        elif r > T2(sum(prob[:max-1]),prob,max):
          df_out.iloc[i,j] = max
        for k in range(1,max-1):
          if T2(sum(prob[:k]),prob,max) <= r < T2(sum(prob[:k+1]),prob,max):
            df_out.iloc[i,j] = k+1
  return df_out

In [48]:
df_origin = unnormal3(df_out,maxs,probs)
df_origin

,0,1,2,3
0,1.0,NaN,1.0,NaN
1,2.0,3.0,1.0,3.0
2,NaN,2.0,NaN,2.0
3,4.0,NaN,3.0,NaN
4,5.0,1.0,5.0,5.0


In [49]:
def unnormal1(df, maxs):
  m = df.shape[0]
  n = df.shape[1]
  df_out = df.copy()
  for i in range(n):
    df_out[i] = df[i]*maxs[i]
  for j in range(n):
    for i in range(m):
      df_out.iloc[i,j] = round(min(maxs[j],max(1, df_out.iloc[i,j])),0)
  return df_out

In [82]:
#compute the sparsity 
n_row = df_origin.shape[0]
n_col = df_origin.shape[1]
Obs_ind = np.where(df_origin.notnull())    # Row and column indices for the observed entries of "Mdat"
num_Obs = len(Obs_ind[0])           # The number of observed entries of "df"
sparsity = 1 - num_Obs / (n_row * n_col)
print("sparsity: "+str(sparsity))

sparsity: 0.30000000000000004


In [51]:
#df_norm = normal1(masked_df)
#df_norm.corr()

In [83]:
corr_mat = df_origin.corr()
corr_mat

,0,1,2,3
0,1.000000,-1.000000,0.953463,1.000000
1,-1.000000,1.000000,-1.000000,-0.654654
2,0.953463,-1.000000,1.000000,1.000000
3,1.000000,-0.654654,1.000000,1.000000


In [84]:
temp_corr = np.copy(corr_mat)
temp_corr[temp_corr < 0] = 0
np.fill_diagonal(temp_corr, 0)
orig_corr = np.copy(temp_corr)
orig_corr

array([[0.        , 0.        , 0.95346259, 1.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.95346259, 0.        , 0.        , 1.        ],
       [1.        , 0.        , 1.        , 0.        ]])

In [85]:
mm = df_origin.shape[0]
nn = df_origin.shape[1]

In [86]:
a_min = 0
a_max = 16
n_level = 4
RMSE_max = 1

In [87]:
masked_df = pd.DataFrame(np.ma.masked_array(df_origin, np.isnan(df)))
masked_df

,0,1,2,3
0,1.0,NaN,1.0,NaN
1,2.0,3.0,1.0,3.0
2,NaN,2.0,NaN,2.0
3,4.0,NaN,3.0,NaN
4,5.0,1.0,5.0,5.0


In [88]:
df_norm = normal1(masked_df)
df_norm

,0,1,2,3
0,0.00,NaN,0.0,NaN
1,0.25,1.0,0.0,0.50
2,NaN,0.5,NaN,0.25
3,0.75,NaN,0.5,NaN
4,1.00,0.0,1.0,1.00


In [89]:
power = nested_quantile(df_norm, a_min, a_max, n_level, RMSE_max)
power

0.5

In [106]:
def c_agg(df, w):
  import numpy as np
  m = df.shape[0]
  n = df.shape[1]

  agg_mat = np.empty(shape=(m, n), dtype='double')
  masked_df = np.ma.masked_array(df, np.isnan(df))
  print(masked_df)

  w = w + 1e-10

  for i in range(len(w)):
    w[i,i]=0
    
  for i in range(m):
    for j in range(n):
      print(masked_df[i,:])
      print(w[:,j])
      agg_mat[i,j] = np.ma.average(masked_df[i,:], weights=w[:,j])
      print(agg_mat[i,j])
  return agg_mat

In [91]:
orig_corr

array([[0.        , 0.        , 0.95346259, 1.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.95346259, 0.        , 0.        , 1.        ],
       [1.        , 0.        , 1.        , 0.        ]])

In [92]:
np.power(orig_corr, power)  

array([[0.        , 0.        , 0.97645409, 1.        ],
       [0.        , 0.        , 0.        , 0.        ],
       [0.97645409, 0.        , 0.        , 1.        ],
       [1.        , 0.        , 1.        , 0.        ]])

In [101]:
np.ma.average([0.25, 1.0, 0.0, 0.5], weights=[0.+1e-10,         0.,         0.+1e-10, 0.+1e-10        ])

0.25

In [107]:
corr = np.power(orig_corr, power)  
agg_mat = c_agg(df_norm, corr)
agg_mat

[[0.0 -- 0.0 --]
 [0.25 1.0 0.0 0.5]
 [-- 0.5 -- 0.25]
 [0.75 -- 0.5 --]
 [1.0 0.0 1.0 1.0]]
[0.0 -- 0.0 --]
[0.0000000e+00 1.0000000e-10 9.7645409e-01 1.0000000e+00]
0.0
[0.0 -- 0.0 --]
[1.e-10 0.e+00 1.e-10 1.e-10]
0.0
[0.0 -- 0.0 --]
[9.7645409e-01 1.0000000e-10 0.0000000e+00 1.0000000e+00]
0.0
[0.0 -- 0.0 --]
[1.e+00 1.e-10 1.e+00 0.e+00]
0.0
[0.25 1.0 0.0 0.5]
[0.0000000e+00 1.0000000e-10 9.7645409e-01 1.0000000e+00]
0.25297830224631873
[0.25 1.0 0.0 0.5]
[1.e-10 0.e+00 1.e-10 1.e-10]
0.25
[0.25 1.0 0.0 0.5]
[9.7645409e-01 1.0000000e-10 0.0000000e+00 1.0000000e+00]
0.3764891511358083
[0.25 1.0 0.0 0.5]
[1.e+00 1.e-10 1.e+00 0.e+00]
0.12500000004375
[-- 0.5 -- 0.25]
[0.0000000e+00 1.0000000e-10 9.7645409e-01 1.0000000e+00]
0.250000000025
[-- 0.5 -- 0.25]
[1.e-10 0.e+00 1.e-10 1.e-10]
0.25
[-- 0.5 -- 0.25]
[9.7645409e-01 1.0000000e-10 0.0000000e+00 1.0000000e+00]
0.250000000025
[-- 0.5 -- 0.25]
[1.e+00 1.e-10 1.e+00 0.e+00]
0.5
[0.75 -- 0.5 --]
[0.0000000e+00 1.0000000e-10 9.7645409

array([[0.        , 0.        , 0.        , 0.        ],
       [0.2529783 , 0.25      , 0.37648915, 0.125     ],
       [0.25      , 0.25      , 0.25      , 0.5       ],
       [0.5       , 0.625     , 0.75      , 0.625     ],
       [1.        , 1.        , 1.        , 1.        ]])

In [108]:
df_agg_mat = pd.DataFrame(agg_mat)
df_agg_mat.columns = ['agg'+str(i) for i in range(df.shape[1])]
df_agg_mat.index = df.index
df_agg = pd.concat([df, df_agg_mat], axis=1)
print(df_agg)

     0    1    2    3      agg0   agg1      agg2   agg3
0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000
1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125
2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500
3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625
4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000


In [109]:
id_column = pd.DataFrame(range(mm))
id_column.index = df_agg.index
print(id_column)
df_agg = pd.concat([id_column, df_agg], axis=1)
df_agg.columns = range(2*nn+1)
print(df_agg)

   0
0  0
1  1
2  2
3  3
4  4
   0    1    2    3    4         5      6         7      8
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000


In [110]:
i=0
df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
df_agg_sorted

,0,1,2,3,4,5,6,7,8
0,0,1.0,NaN,1.0,NaN,0.000000,0.000,0.000000,0.000
2,2,NaN,2.0,NaN,2.0,0.250000,0.250,0.250000,0.500
1,1,2.0,3.0,1.0,3.0,0.252978,0.250,0.376489,0.125
3,3,4.0,NaN,3.0,NaN,0.500000,0.625,0.750000,0.625
4,4,5.0,1.0,5.0,5.0,1.000000,1.000,1.000000,1.000


In [111]:
index_i = df_agg_sorted.index
vec = np.array(df_agg_sorted.iloc[:,i+1])
vec

array([ 1., nan,  2.,  4.,  5.])

In [112]:
out = monotonization(vec)
out

array([1., 2., 2., 4., 5.])

In [113]:
out_list = []
for i in range(len(out)):
    out_list.append(out[i])

out_df = pd.DataFrame(out_list, index=index_i)
df_agg = pd.concat([df_agg_sorted, out_df], axis=1)
df_agg

,0,1,2,3,4,5,6,7,8,0
0,0,1.0,NaN,1.0,NaN,0.000000,0.000,0.000000,0.000,1.0
2,2,NaN,2.0,NaN,2.0,0.250000,0.250,0.250000,0.500,2.0
1,1,2.0,3.0,1.0,3.0,0.252978,0.250,0.376489,0.125,2.0
3,3,4.0,NaN,3.0,NaN,0.500000,0.625,0.750000,0.625,4.0
4,4,5.0,1.0,5.0,5.0,1.000000,1.000,1.000000,1.000,5.0


In [114]:
i=1
df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
print(df_agg_sorted)
index_i = df_agg_sorted.index
vec = np.array(df_agg_sorted.iloc[:,i+1])
print(vec)
out = monotonization(vec)
print(out)
out_list = []
for i in range(len(out)):
    out_list.append(out[i])

out_df = pd.DataFrame(out_list, index=index_i)
df_agg = pd.concat([df_agg_sorted, out_df], axis=1)
print(df_agg)

   0    1    2    3    4         5      6         7      8    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0
[nan  2.  3. nan  1.]
[2. 2. 3. 3. 1.]
   0    1    2    3    4         5      6         7      8    0    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0  2.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0  2.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0  3.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0  3.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0  1.0


In [115]:
i=2
df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
print(df_agg_sorted)
index_i = df_agg_sorted.index
vec = np.array(df_agg_sorted.iloc[:,i+1])
print(vec)
out = monotonization(vec)
print(out)
out_list = []
for i in range(len(out)):
    out_list.append(out[i])

out_df = pd.DataFrame(out_list, index=index_i)
df_agg = pd.concat([df_agg_sorted, out_df], axis=1)
print(df_agg)

   0    1    2    3    4         5      6         7      8    0    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0  2.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0  2.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0  3.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0  3.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0  1.0
[ 1. nan  1.  3.  5.]
[1. 1. 1. 3. 5.]
   0    1    2    3    4         5      6         7      8    0    0    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0  2.0  1.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0  2.0  1.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0  3.0  1.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0  3.0  3.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0  1.0  5.0


In [116]:
i=3
df_agg_sorted = df_agg.sort_values(by=[df_agg.columns[nn+i+1],df_agg.columns[i+1]])
print(df_agg_sorted)
index_i = df_agg_sorted.index
vec = np.array(df_agg_sorted.iloc[:,i+1])
print(vec)
out = monotonization(vec)
print(out)
out_list = []
for i in range(len(out)):
    out_list.append(out[i])

out_df = pd.DataFrame(out_list, index=index_i)
df_agg = pd.concat([df_agg_sorted, out_df], axis=1)
print(df_agg)

   0    1    2    3    4         5      6         7      8    0    0    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0  2.0  1.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0  3.0  1.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0  2.0  1.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0  3.0  3.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0  1.0  5.0
[nan  3.  2. nan  5.]
[3. 3. 2. 5. 5.]
   0    1    2    3    4         5      6         7      8    0    0    0    0
0  0  1.0  NaN  1.0  NaN  0.000000  0.000  0.000000  0.000  1.0  2.0  1.0  3.0
1  1  2.0  3.0  1.0  3.0  0.252978  0.250  0.376489  0.125  2.0  3.0  1.0  3.0
2  2  NaN  2.0  NaN  2.0  0.250000  0.250  0.250000  0.500  2.0  2.0  1.0  2.0
3  3  4.0  NaN  3.0  NaN  0.500000  0.625  0.750000  0.625  4.0  3.0  3.0  5.0
4  4  5.0  1.0  5.0  5.0  1.000000  1.000  1.000000  1.000  5.0  1.0  5.0  5.0
